# V5 SE-OI: piloto CPU con guardado automático en GitHub

**Esta versión utiliza GitHub y una carpeta local de Colab. No monta Drive.**

Candidata fijada: `seoi_mse`; seis recetas por cinco semillas,30 ajustes nuevos. El notebook se entrega sin entrenamientos ejecutados. La metodología y su regla de decisión se conservan.

Los resultados y recibos son legibles en la rama de ejecución. Los checkpoints con pesos, Adam y RNG se guardan **cifrados**. El archivo NPZ de datos no se publica de nuevo.

Necesitas un token de GitHub para `jassan-bit/DeepLearning_202630` con **Contents: Read and write**, y una clave privada de respaldo de al menos32 caracteres. Guárdalos en los secretos de Colab con los nombres `GITHUB_TOKEN` y `E3_V5_BACKUP_KEY`, o introdúcelos en los campos privados. Conserva la clave para recuperar checkpoints; no la pegues en el chat.

No ejecutes dos runtimes contra la misma rama. Para otra ejecución usa otra `RUN_BRANCH` antes de preparar el caso. Para reanudar, conserva rama, versión y clave.


## 1. Instala y verifica; cero entrenamientos
Runtime CPU. Fuentes y datos fijados por commit y SHA256; dependencias en un entorno separado del Torch de Colab. La preparación ejecuta las pruebas sintéticas del programa.


In [ ]:
import hashlib
import urllib.request

SOURCE_COMMIT = 'c4ca86f1431afeaa958ff0b544b453ab17a4d7b8'
SOURCE_MANIFEST_SHA256 = '4706f20a00508e492a54e25873b4885cead3fa7660d43bdb18c5aeda4fce2509'
BOOTSTRAP_SHA256 = '3a6bce99e149e3a1e6cfbb07b8e23fb6482d9bb876597d4f8d5df198cde07941'
REPOSITORY = "jassan-bit/DeepLearning_202630"
RUN_BRANCH = "e3-seoi-v5-run-01"
RESULT_PREFIX = "runs/e3_seoi_v5/run-01"
CASE_ROOT = "/content/e3-seoi-v5-cases/" + hashlib.sha256((REPOSITORY + RUN_BRANCH).encode()).hexdigest()[:16]
url = (
    "https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/"
    + SOURCE_COMMIT
    + "/experiments/e3_seoi_v5_pilot_20261121/bootstrap_colab.py"
)
with urllib.request.urlopen(url, timeout=60) as response:
    code = response.read(2 * 1024 * 1024)
if hashlib.sha256(code).hexdigest() != BOOTSTRAP_SHA256:
    raise RuntimeError("El código descargado no coincide con la versión fijada.")
E3_MODULE = {"__name__": "e3_v5_github_setup"}
exec(compile(code, "e3_v5_github_setup", "exec"), E3_MODULE)
E3 = E3_MODULE["setup"](
    source_commit=SOURCE_COMMIT,
    manifest_sha256=SOURCE_MANIFEST_SHA256,
    case_root=CASE_ROOT,
    prepare_case=False,
)
print("Entorno verificado. La celda siguiente guardará o restaurará el caso en GitHub.")


## 2. Acceso privado y primer guardado/restauración
Verifica el guardado antes de permitir entrenamientos. Si la rama ya contiene este caso, recupera su respaldo autenticado. Las credenciales viajan por stdin al proceso y se excluyen de archivos y argumentos.


In [ ]:
from getpass import getpass

def private_value(name, prompt):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = getpass(prompt)
    if not isinstance(value, str) or not value.strip():
        raise ValueError("Falta la credencial privada solicitada.")
    return value.strip()

E3_CREDENTIALS = {
    "token": private_value("GITHUB_TOKEN", "Token GitHub con Contents: Read and write: "),
    "backup_secret": private_value("E3_V5_BACKUP_KEY", "Clave privada de respaldo (mínimo32caracteres; consérvala): "),
}
if len(E3_CREDENTIALS["backup_secret"]) < 32:
    raise ValueError("La clave de respaldo debe tener al menos32caracteres y conservarse para reanudar.")

def E3_GITHUB(action):
    return E3_MODULE["github_command"](
        E3, action, source_commit=SOURCE_COMMIT,
        token=E3_CREDENTIALS["token"], backup_secret=E3_CREDENTIALS["backup_secret"],
        repository=REPOSITORY, branch=RUN_BRANCH, prefix=RESULT_PREFIX,
    )

E3_GITHUB("setup")
print("Caso verificado en GitHub. No se ha iniciado entrenamiento en estas dos celdas.")


## 3. Ejecuta o reanuda
Cada trabajo terminado se respalda y verifica antes del siguiente. También se guardan checkpoints intermedios al alcanzar un minuto entre respaldos. El cliente espacia las escrituras y distingue límites temporales de permisos faltantes.


In [ ]:
# Inicia o reanuda el piloto. Cada trabajo completo se guarda antes del siguiente.
# Si un guardado falla, se detienen nuevos ajustes y se conserva el estado local.
E3_GITHUB("run")


## 4. Evalúa y guarda los resultados
Se requieren los 30trabajos completos. Regla fija: mejora RMSE≥2% frente al mejor control,≥4/5 victorias frente al mejor control neuronal por RMSE y MAE≤1,05veces la de ese control. No cambia de candidata después de ver el resultado.


In [ ]:
# Exige30trabajos completos. Guarda tablas, decisión y respaldo en GitHub.
E3_GITHUB("assess")


## 5. Comprueba GitHub
`status.json` resume el avance; `public_results.zip` conserva tablas y recibos; `checkpoint.aesgcm` contiene el respaldo cifrado. Los commits de guardado aparecen en la salida de las celdas.


In [ ]:
from IPython.display import display, Markdown
E3_GITHUB("status")
results_url = "https://github.com/" + REPOSITORY + "/tree/" + RUN_BRANCH + "/" + RESULT_PREFIX
display(Markdown("[Abrir resultados y recibos en GitHub](" + results_url + ")"))


## Recuperación y alcance

Tras una desconexión, ejecuta 1 y 2 con la misma rama, versión y clave, y después 3. Los trabajos guardados se verifican y se omiten. El progreso posterior al último respaldo puede perderse.

Si se restaura una sesión abierta, se conserva esa incertidumbre en `recovery_history.json`: los pasos y costes registrados son una cota inferior del cómputo total. No se inventa un consumo exacto ni se transforma el presupuesto nominal 120 s en un límite histórico garantizado. Las métricas de desarrollo se conservan con ese alcance.

El piloto usa 55 fechas de desarrollo previamente expuestas. Los 796trabajos aditivos y los pilotos V3/V4 permanecen separados. La superioridad requiere resultados experimentales y la confirmación descrita en el protocolo; las pruebas del programa no la acreditan.

Asistencia de IA declarada en formulación, literatura, programación y pruebas. Las celdas de entrenamiento/evaluación se entregan sin ejecutar.
